# Load images

In [ ]:
import os

folder_path = r'C:\Users\ck\Downloads\CellData\OCT'
image_extensions = ['.jpeg']

image_files = []

print("Initialized variables and an empty list for image files.")

Initialized variables and an empty list for image files.


In [ ]:
import os
import cv2
import numpy as np

# Define the output directory for cropped images
cropped_output_directory_name = 'OCT_cropped'
cropped_output_path = os.path.join(os.path.dirname(folder_path), cropped_output_directory_name)
os.makedirs(cropped_output_path, exist_ok=True)

print(f"输出裁剪图像的目录 '{cropped_output_path}' 已确保存在。")

实现 `crop_white_border` 辅助函数，该函数将利用 `OpenCV` 检测并裁剪图像中的白色边框。该函数应能够处理灰度图像和彩色图像，并通过查找非白色区域的边界框来工作。它将返回裁剪后的图像。

In [ ]:
def crop_white_border(image, border_color=(255, 255, 255), tolerance=10):
    """
    裁剪图像中的白色（或指定 border_color）边框。

    参数:
        image (np.array): 输入图像 (OpenCV 格式)。
        border_color (tuple): 视为边框的颜色 (例如，(255, 255, 255) 表示白色)。
        tolerance (int): 颜色匹配的容差 (例如，10 表示像素值在 border_color 10 范围内都视为边框)。

    返回:
        np.array: 裁剪后的图像。
    """
    if image is None:
        return None

    # 判断图像是灰度图还是彩图
    is_grayscale = (len(image.shape) == 2 or (len(image.shape) == 3 and image.shape[2] == 1))

    if is_grayscale:
        # 对于灰度图，将 border_color 元组转换为单个值
        border_value = border_color[0] # 假设白色边框，取第一个值
        # 创建一个掩码，其中边框像素为 0，内容为 255
        mask = cv2.inRange(image, border_value - tolerance, border_value + tolerance)
        mask = cv2.bitwise_not(mask) # 反转掩码以获取内容
    else:
        # 对于彩图，处理每个通道
        lower_bound = np.array([c - tolerance for c in border_color])
        upper_bound = np.array([c + tolerance for c in border_color])
        mask = cv2.inRange(image, lower_bound, upper_bound)
        mask = cv2.bitwise_not(mask) # 反转掩码以获取内容

    # 查找轮廓
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return image # 未找到内容，返回原始图像

    # 获取最大轮廓的边界框 (假设内容是最大的非边框区域)
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)

    # 裁剪图像
    cropped_image = image[y:y+h, x:x+w]

    return cropped_image

print("`crop_white_border` 函数已定义。")


遍历 `image_files` 列表中的每个图像，加载它，应用 `crop_white_border` 函数来裁剪白色边框，并将裁剪后的图像保存到 `OCT_cropped` 目录中，同时保留原始文件夹结构。在处理过程中提供进度更新，并在完成后提供摘要。

In [ ]:
cropped_count = 0
skipped_count = 0

print(f"开始裁剪图像，共 {len(image_files)} 个文件...")

for i, filepath in enumerate(image_files):
    # 构建相对路径
    relative_path = os.path.relpath(filepath, folder_path)

    # 构建 cropped_output_path 内的完整目标目录路径
    target_directory = os.path.join(cropped_output_path, os.path.dirname(relative_path))

    # 如果此目标目录不存在，则创建它
    os.makedirs(target_directory, exist_ok=True)

    try:
        # 使用 cv2.imread 加载图像
        original_image = cv2.imread(filepath)

        if original_image is None:
            print(f"警告: 无法加载图像文件 - {filepath}。跳过。")
            skipped_count += 1
            continue

        # 应用 crop_white_border 函数
        cropped_image = crop_white_border(original_image)

        if cropped_image is None or cropped_image.size == 0:
            print(f"警告: 裁剪失败或未找到内容 - {filepath}。跳过。")
            skipped_count += 1
            continue

        # 构建保存的唯一文件名
        base_filename = os.path.basename(filepath)
        name_without_ext, ext = os.path.splitext(base_filename)
        cropped_save_path = os.path.join(target_directory, f'{name_without_ext}_cropped{ext}')

        # 保存裁剪后的图像
        cv2.imwrite(cropped_save_path, cropped_image)

        cropped_count += 1

    except Exception as e:
        print(f"处理图像 {filepath} 时出错: {e}")
        skipped_count += 1

    # 定期打印进度消息
    if (i + 1) % 10000 == 0:
        print(f"已处理 {i + 1}/{len(image_files)} 张图像进行裁剪...")

print(f"\n裁剪完成。总共处理并裁剪的图像数量: {cropped_count}。")
print(f"由于加载问题或无内容而跳过的图像数量: {skipped_count}。")
print(f"所有裁剪后的图像都保存在 '{cropped_output_path}' 中。")

In [ ]:
for root, _, files in os.walk(folder_path):
    for file in files:
        if any(file.lower().endswith(ext) for ext in image_extensions):
            image_files.append(os.path.join(root, file))

print(f"Found {len(image_files)} image files.")

Found 109309 image files.


## 统计图片分辨率

统计并报告所有图像文件的不同分辨率数量。

为了统计不同分辨率的数量，我将使用 `Pillow` 库来打开每个唯一的图像文件并获取其尺寸。我会从 `hashes_map` 中提取唯一的图片路径，然后将所有分辨率存储在一个列表中，最后使用 `collections.Counter` 来统计每种分辨率的图片数量，并报告总数和每种分辨率的详细信息。

In [ ]:
from PIL import Image
from collections import Counter
import os

image_resolutions = []

# Get a list of unique image file paths (one from each hash group)
# This prevents processing the same image content multiple times for resolution
unique_image_paths = [paths[0] for paths in hashes_map.values()]

print(f"Extracting resolutions from {len(unique_image_paths)} unique image files...")

for i, filepath in enumerate(unique_image_paths):
    try:
        with Image.open(filepath) as img:
            width, height = img.size
            image_resolutions.append((width, height))
    except FileNotFoundError:
        print(f"Warning: File not found - {filepath}. Skipping resolution extraction.")
    except Image.UnidentifiedImageError:
        print(f"Warning: Could not identify image file - {filepath}. Skipping resolution extraction.")
    except Exception as e:
        print(f"Error processing image {filepath}: {e}")

    if (i + 1) % 10000 == 0:
        print(f"Processed {i + 1}/{len(unique_image_paths)} unique files for resolution...")

# Count the occurrences of each unique resolution
resolution_counts = Counter(image_resolutions)

if resolution_counts:
    print(f"\nFound {len(resolution_counts)} unique resolutions:")
    for resolution, count in resolution_counts.most_common():
        print(f"- {resolution[0]}x{resolution[1]}: {count} images")
else:
    print("No resolutions could be extracted from the image files.")

Extracting resolutions from 14772 unique image files...
Processed 10000/14772 unique files for resolution...

Found 7 unique resolutions:
- 512x496: 7340 images
- 768x496: 4458 images
- 1536x496: 1963 images
- 1024x496: 1003 images
- 384x496: 6 images
- 642x277: 1 images
- 900x455: 1 images


To count the images per disease type, I will iterate through the `image_files` list. For each file path, I'll extract the disease category by looking for a common pattern (e.g., the directory name before the image file). Then, I'll use `collections.Counter` to tally the number of images for each identified disease type and print the results.

In [ ]:
from collections import Counter
import os

disease_types = []

# Extract disease type from each file path
for filepath in image_files:
    # Assuming the disease type is the directory name right before the image file
    # e.g., 'C:\Users\ck\Downloads\CellData\OCT\train\CNV\image.jpeg' -> 'CNV'
    parts = filepath.split(os.sep)
    # Try to find common disease types like 'CNV', 'DME', 'DRUSEN', 'NORMAL' in the path
    # This part might need adjustment based on the exact path structure if not consistent
    if 'CNV' in parts:
        disease_types.append('CNV')
    elif 'DME' in parts:
        disease_types.append('DME')
    elif 'DRUSEN' in parts:
        disease_types.append('DRUSEN')
    elif 'NORMAL' in parts:
        disease_types.append('NORMAL')
    # Add more conditions here if other disease types exist in the path structure

# Count the occurrences of each disease type
disease_counts = Counter(disease_types)

if disease_counts:
    print("Image counts per disease type:")
    for disease, count in disease_counts.items():
        print(f"- {disease}: {count} images")
else:
    print("Could not identify disease types from file paths or no images found.")

Image counts per disease type:
- CNV: 37455 images
- DME: 11598 images
- DRUSEN: 8866 images
- NORMAL: 51390 images


To confirm that no images have the same name, first extract the base filename from each path in `image_files`. Then, use a `Counter` to efficiently identify and count any duplicate filenames. Finally, I'll report the filenames that appear more than once.

In [ ]:
from collections import Counter
import os

# Extract just the filenames from the full paths
filenames = [os.path.basename(filepath) for filepath in image_files]

# Count the occurrences of each filename
filename_counts = Counter(filenames)

# Identify duplicate filenames (those with count > 1)
duplicate_filenames = {filename: count for filename, count in filename_counts.items() if count > 1}

if duplicate_filenames:
    print(f"Found {len(duplicate_filenames)} duplicate filenames:")
    for filename, count in duplicate_filenames.items():
        print(f"- '{filename}' appears {count} times")
    print("\nAction Required: You may need to rename these files or investigate why they are duplicated.")
else:
    print("No duplicate filenames found. All image files have unique names.")

No duplicate filenames found. All image files have unique names.


## Prepare for Augmentation

### Subtask:
Define the parameters for data augmentation, specifically the maximum pixel shift for translation and the fill color (white) for new areas. Also, set up and ensure the target output directory `OCT_clean` exists to store the augmented images.


**Reasoning**:
To prepare for data augmentation, I will define the `max_pixel_shift` and `fill_color_white` parameters, set the `output_directory_name` to 'OCT_clean', and then use the `os` module to ensure this output directory exists, creating it if necessary.



## Prepare for Augmentation

### Subtask:
Define the parameters for data augmentation, including the minimum and maximum percentage for random pixel shifts (5% to 20% of image dimensions) and the fill color (white) for new areas. Also, set up and ensure the target output directory `OCT_clean` exists to store the augmented images.


**Reasoning**:
I will define the augmentation parameters: minimum and maximum pixel shift percentages, and the fill color. Then, I will ensure the target output directory exists by creating it if it doesn't already.



In [ ]:
import os

# 1. Define minimum and maximum percentage for random pixel shifts
min_shift_percentage = 0.05  # 5%
max_shift_percentage = 0.20  # 20%

# 2. Define the fill color (white) for new areas
fill_color_white = (255, 255, 255)

# 3. Set the target output directory name
output_directory_name = 'OCT_clean'

# 4. Create the output directory if it doesn't exist
output_path = os.path.join(os.path.dirname(folder_path), output_directory_name)
os.makedirs(output_path, exist_ok=True)

print(f"Augmentation parameters defined: min_shift_percentage={min_shift_percentage}, max_shift_percentage={max_shift_percentage}, fill_color={fill_color_white}")
print(f"Output directory '{output_path}' ensured to exist.")

Augmentation parameters defined: min_shift_percentage=0.05, max_shift_percentage=0.2, fill_color=(255, 255, 255)
Output directory 'C:\Users\ck\Downloads\CellData\OCT_clean' ensured to exist.


## Process and Augment Images

### Subtask:
Iterate through each image in the `image_files` list, load it, apply a random translation based on defined pixel shift percentages, resize both the original and translated images to 224x224, and save them into the `OCT_clean` directory, mirroring the original folder structure.


**Reasoning**:
First, I will define a helper function `translate_image` to perform random translations on an image. This function will calculate random pixel shifts based on the defined percentages and apply an affine transformation. Then, I will initialize counters for processed and augmented images.



**Reasoning**:
Now, I will iterate through each image file, load it, apply the defined random translation, resize both the original and translated images to 224x224, and then save them into the `OCT_clean` directory, preserving the original folder structure, and finally print a summary of the processed images.



# Task
Modify the `translate_image` helper function to handle both grayscale and color images by dynamically setting the `fillcolor` based on the image mode. Then, re-execute the image processing and augmentation loop using this updated function. Finally, summarize the total number of original images processed and augmented images generated, confirming that all new images are stored in the `OCT_clean` folder and that the previous errors related to `fillcolor` have been resolved.

## Modify Translate Image Function

### Subtask:
Adjust the `translate_image` helper function to dynamically determine the appropriate `fillcolor` based on the input image's mode. If the image is grayscale ('L' mode), the `fillcolor` should be a single integer (e.g., 255 for white). For color images (e.g., 'RGB'), the current tuple `(255, 255, 255)` can be used.


**Reasoning**:
The subtask requires modifying the `translate_image` function to dynamically set the `fillcolor` based on the image's mode. I will update the function definition to remove the `fill_color_white` parameter and implement the logic to choose between a single integer for grayscale images and a tuple for color images.



# Task
Generate augmented images by applying random pixel shifts to the original dataset using OpenCV. Define augmentation parameters (5-20% shift, white fill color) and ensure the output directory `OCT_clean` exists. Implement a `translate_image` function using `cv2.warpAffine` that dynamically handles fill colors for grayscale and color images. Iterate through all original images, load them with OpenCV, apply the translation, resize both original and augmented images to 224x224, and save them into the `OCT_clean` directory while maintaining the original folder structure. Finally, provide a summary of the total original and augmented images, confirming their storage location and the resolution of previous `fillcolor` errors.

## Prepare for Augmentation (OpenCV)

### Subtask:
Define the parameters for data augmentation, including the minimum and maximum percentage for random pixel shifts (5% to 20% of image dimensions) and the fill color (white) for new areas. Also, set up and ensure the target output directory `OCT_clean` exists to store the augmented images. This step will also import `cv2` and `numpy` for subsequent image processing.


**Reasoning**:
I need to import the `cv2` and `numpy` libraries, define the specified augmentation parameters, and ensure the output directory `OCT_clean` is set up, as requested by the subtask.



In [ ]:
import cv2
import numpy as np
import os

# 1. Define minimum and maximum percentage for random pixel shifts
min_shift_percentage = 0.05  # 5%
max_shift_percentage = 0.20  # 20%

# 2. Define the fill color (white) for new areas (for color images)
fill_color_white = (255, 255, 255)

# 3. Set the target output directory name
output_directory_name = 'OCT_clean'

# Assuming 'folder_path' is defined in a previous cell, e.g., folder_path = r'C:\Users\ck\Downloads\CellData\OCT'
# Ensure folder_path is accessible, or redefine it if this cell is meant to be standalone for these variables.
# For continuity, we will assume folder_path is available from previous cells.

# 4. Construct the full path for the output directory
output_path = os.path.join(os.path.dirname(folder_path), output_directory_name)

# 5. Create the output directory if it doesn't exist
os.makedirs(output_path, exist_ok=True)

print(f"`cv2` and `numpy` imported. Augmentation parameters defined: min_shift_percentage={min_shift_percentage}, max_shift_percentage={max_shift_percentage}, fill_color_white={fill_color_white}.")
print(f"Output directory '{output_path}' ensured to exist.")

`cv2` and `numpy` imported. Augmentation parameters defined: min_shift_percentage=0.05, max_shift_percentage=0.2, fill_color_white=(255, 255, 255).
Output directory 'C:\Users\ck\Downloads\CellData\OCT_clean' ensured to exist.


## Implement translate_image with OpenCV

### Subtask:
Define a helper function `translate_image` that utilizes `OpenCV` for image translation. This function will take an image (NumPy array), min/max shift percentages, and the fill color. It will calculate random pixel shifts, apply an affine transformation using `cv2.warpAffine`, and dynamically determine the `fillcolor` based on whether the image is grayscale or color. It will return the translated image as a NumPy array.


**Reasoning**:
I need to define the `translate_image` helper function using OpenCV, ensuring it handles dynamic fill colors for grayscale and color images, calculates random shifts, and applies affine transformation. After defining the function, I will initialize the necessary counters as per the instructions.



In [ ]:
import random
import cv2
import numpy as np

def translate_image(image, min_shift_percentage, max_shift_percentage, fill_color_white):
    # 1. Get image dimensions
    if len(image.shape) == 3:
        height, width, _ = image.shape
    else:
        height, width = image.shape

    # 2. Calculate min/max pixel shifts
    min_shift_pixels_x = int(width * min_shift_percentage)
    max_shift_pixels_x = int(width * max_shift_percentage)
    min_shift_pixels_y = int(height * min_shift_percentage)
    max_shift_pixels_y = int(height * max_shift_percentage)

    # 3. Ensure min shift is at least 1 pixel if percentage is greater than 0
    if min_shift_pixels_x == 0 and min_shift_percentage > 0: min_shift_pixels_x = 1
    if min_shift_pixels_y == 0 and min_shift_percentage > 0: min_shift_pixels_y = 1

    # 4. Generate random dx (horizontal shift)
    dx_magnitude = random.randint(min_shift_pixels_x, max_shift_pixels_x)
    dx = dx_magnitude if random.choice([True, False]) else -dx_magnitude

    # 5. Generate random dy (vertical shift)
    dy_magnitude = random.randint(min_shift_pixels_y, max_shift_pixels_y)
    dy = dy_magnitude if random.choice([True, False]) else -dy_magnitude

    # 6. Create the 2x3 affine transformation matrix M for translation
    M = np.float32([[1, 0, dx], [0, 1, dy]])

    # 7. Dynamically determine the fill_value for cv2.warpAffine
    if len(image.shape) == 3:  # Color image (e.g., RGB)
        fill_value = fill_color_white # tuple (255, 255, 255)
    else:  # Grayscale image (e.g., L)
        fill_value = fill_color_white[0] # single value from the tuple, e.g., 255

    # 8. Apply the affine transformation
    translated_image = cv2.warpAffine(image, M, (width, height), borderValue=fill_value)

    return translated_image


processed_images_count = 0
augmented_images_count = 0

print("Helper function `translate_image` defined using OpenCV and counters initialized.")

Helper function `translate_image` defined using OpenCV and counters initialized.


## Process and Augment Images with OpenCV

### Subtask:
Iterate through each image in the `image_files` list one by one. For each image, load it using `cv2.imread`, convert it to the appropriate color space if necessary (e.g., from BGR to RGB if using PIL later, or keep BGR for OpenCV processing), apply the `translate_image` function, resize both the original and translated images to 224x224 using `cv2.resize`. Save these processed images into appropriate subfolders within the `OCT_clean` directory, mirroring the original dataset's structure, and ensure unique filenames for augmented images. Error handling will be included for robust processing.


**Reasoning**:
I will iterate through each image file, load it using OpenCV, apply the `translate_image` function, resize both the original and translated images to 224x224, and save them to the `OCT_clean` directory while maintaining the original folder structure. I will also include error handling and print progress updates and a final summary of the processed images.



In [ ]:
for i, filepath in enumerate(image_files):
    # 1. Construct the relative path
    relative_path = os.path.relpath(filepath, folder_path)

    # 2. Construct the full target directory path within output_path
    target_directory = os.path.join(output_path, os.path.dirname(relative_path))

    # 3. Create this target directory if it doesn't exist
    os.makedirs(target_directory, exist_ok=True)

    try:
        # 4. Load the image using cv2.imread
        original_image_bgr = cv2.imread(filepath)

        if original_image_bgr is None:
            print(f"Warning: Could not load image file - {filepath}. Skipping.")
            continue

        # OpenCV loads images in BGR format. If the image is grayscale, it will be 2D.
        # The translate_image function is designed to handle both.

        # 5. Call translate_image helper function
        translated_image_bgr = translate_image(original_image_bgr.copy(), min_shift_percentage, max_shift_percentage, fill_color_white)

        # 6. Resize the original and translated images to (224, 224)
        resized_original_image = cv2.resize(original_image_bgr, (224, 224), interpolation=cv2.INTER_AREA)
        resized_translated_image = cv2.resize(translated_image_bgr, (224, 224), interpolation=cv2.INTER_AREA)

        # 7. Extract the base filename and extension
        base_filename = os.path.basename(filepath)
        name_without_ext, ext = os.path.splitext(base_filename)

        # 8. Construct unique filenames for saving
        original_save_path = os.path.join(target_directory, f'{name_without_ext}_original{ext}')
        augmented_save_path = os.path.join(target_directory, f'{name_without_ext}_augmented{ext}')

        # 9. Save both resized images
        cv2.imwrite(original_save_path, resized_original_image)
        cv2.imwrite(augmented_save_path, resized_translated_image)

        # 10. Increment counters
        processed_images_count += 1
        augmented_images_count += 1 # Each original image produces one augmented image

    except Exception as e:
        print(f"Error processing image {filepath}: {e}")

    # 11. Periodically print a progress message
    if (i + 1) % 10000 == 0:
        print(f"Processed {i + 1}/{len(image_files)} images...")

# 12. Print final summary
print(f"\nFinished processing. Total original images processed: {processed_images_count}.")
print(f"Total augmented images generated: {augmented_images_count}.")

Processed 10000/109309 images...
Processed 20000/109309 images...
Processed 30000/109309 images...
Processed 40000/109309 images...
Processed 50000/109309 images...
Processed 60000/109309 images...
Processed 70000/109309 images...
Processed 80000/109309 images...
Processed 90000/109309 images...
Processed 100000/109309 images...

Finished processing. Total original images processed: 109309.
Total augmented images generated: 109309.


## Final Task

### Subtask:
Summarize the data augmentation process, including the total number of original images processed and augmented images generated, confirm that all new images are stored in the `OCT_clean` folder as specified, and confirm that the previous errors related to `fillcolor` have been resolved by using `OpenCV`.


## Summary:

### Q&A

1.  **Summarize the data augmentation process:**
    The data augmentation process involved applying random pixel shifts (5-20% of image dimensions) to original images using `cv2.warpAffine` from the OpenCV library. A white fill color was used for new areas created by the shifts. Both the original and augmented images were then resized to 224x224 pixels and saved into the `OCT_clean` directory, mirroring the original dataset's folder structure.

2.  **Total number of original images processed and augmented images generated:**
    A total of 109,309 original images were processed, and 109,309 augmented images were generated.

3.  **Confirm that all new images are stored in the `OCT_clean` folder as specified:**
    Yes, all new original (`_original`) and augmented (`_augmented`) images were successfully stored in the `OCT_clean` directory, which was created at `C:\Users\ck\Downloads\CellData\OCT_clean`.

4.  **Confirm that the previous errors related to `fillcolor` have been resolved by using `OpenCV`:**
    Yes, the `fillcolor` errors were resolved. The `translate_image` function dynamically determines the `fill_value` for `cv2.warpAffine`: it uses the full `(255, 255, 255)` tuple for color images and only the first channel's value (255) for grayscale images, ensuring correct white fill for both image types.

### Data Analysis Key Findings

*   **Augmentation Parameters:** Random pixel shifts were applied within a range of 5% to 20% of the image's dimensions.
*   **Fill Color Handling:** A white fill color (255, 255, 255) was successfully used, with dynamic adaptation for grayscale images (using 255), resolving previous `fillcolor` issues.
*   **Image Processing Count:** The process successfully handled 109,309 original images, generating an equal number of 109,309 augmented images.
*   **Image Resizing:** All original and augmented images were consistently resized to a dimension of 224x224 pixels.
*   **Output Location:** All processed images (both original and augmented versions) were stored in the newly created `OCT_clean` directory, maintaining the original dataset's hierarchical structure.

### Insights or Next Steps

*   The successful augmentation of a large dataset (109,309 images) with robust error handling and correct `fillcolor` management significantly expands the training data, which could lead to improved model generalization and performance.
*   The creation of both `_original` and `_augmented` versions within `OCT_clean` allows for flexible use in training, either by combining them or selecting specific subsets, and sets a clean, standardized dataset for subsequent machine learning tasks.
